# ISL Sign Recognition — Improved Training for 81%+ Accuracy (263 classes)

**Target model:** `include_no_cnn_transformer_small.pth`

**Improvements over original:**
- AdamW optimizer with weight decay
- Warmup + CosineAnnealing LR scheduler
- Higher augmentation probability (0.5 vs 0.4)
- Gradient clipping
- More epochs (75) with higher patience (15)
- Mixup augmentation on tensors

**Branch:** `cnn_new` | **Time:** ~60 min on Colab GPU

In [ ]:
# ── Step 1: Clone cnn_new branch ─────────────────────────────────────
!git clone --branch cnn_new --single-branch https://github.com/BishalDubey27/Major_Project.git
%cd Major_Project
!ls

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────────────
!pip install mediapipe==0.10.31 transformers==4.44.0 timm joblib tqdm scikit-learn -q
print('Done')

In [ ]:
# ── Step 3: Load keypoints from Google Drive (3 separate folders) ─────
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

# ── Change these paths to match your Drive folder locations ──────────
DRIVE_TRAIN = '/content/drive/MyDrive/include_train_keypoints'
DRIVE_VAL   = '/content/drive/MyDrive/include_val_keypoints'
DRIVE_TEST  = '/content/drive/MyDrive/include_test_keypoints'

os.makedirs('/content/keypoint', exist_ok=True)
shutil.copytree(DRIVE_TRAIN, '/content/keypoint/include_train_keypoints')
shutil.copytree(DRIVE_VAL,   '/content/keypoint/include_val_keypoints')
shutil.copytree(DRIVE_TEST,  '/content/keypoint/include_test_keypoints')

for split in ['include_train_keypoints', 'include_val_keypoints', 'include_test_keypoints']:
    n = len([f for f in os.listdir(f'/content/keypoint/{split}') if f.endswith('.json')])
    print(f'{split}: {n} files')

In [ ]:
# ── Step 4: Improved training pipeline ───────────────────────────────
import os, sys, json, math
import torch
import torch.nn.functional as F
from torch.utils import data as torch_data
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import numpy as np

sys.path.insert(0, '/content/Major_Project/INCLUDE')
sys.path.insert(0, '/content/Major_Project')

from models.transformer import Transformer
from configs import TransformerConfig
from dataset import KeypointsDataset
from utils import seed_everything, AverageMeter, EarlyStopping, load_json

os.chdir('/content/Major_Project/INCLUDE')

# ── Config ────────────────────────────────────────────────────────────
DATASET       = 'include'       # 263 classes
DATA_DIR      = '/content/keypoint'
SAVE_PATH     = '/content'
EPOCHS        = 75
BATCH_SIZE    = 128
LR            = 1e-4
WEIGHT_DECAY  = 1e-2
WARMUP_EPOCHS = 5
PATIENCE      = 15
GRAD_CLIP     = 1.0
MIXUP_ALPHA   = 0.2             # 0 to disable mixup
SEED          = 42
SIZE          = 'small'         # 'small' or 'large' for more capacity

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
seed_everything(SEED)

label_map = load_json(f'label_maps/label_map_{DATASET}.json')
n_classes = len(label_map)
print(f'Classes: {n_classes}')

# ── Model ─────────────────────────────────────────────────────────────
config = TransformerConfig(size=SIZE)
model  = Transformer(config=config, n_classes=n_classes).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

# ── Dataset with higher aug probability ───────────────────────────────
from augment import Augmentation, OneOf, plus7rotation, minus7rotation, gaussSample, cutout, upsample, downsample

class ImprovedKeypointsDataset(KeypointsDataset):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Higher augmentation probability
        self.augs = [
            Augmentation(OneOf(plus7rotation, minus7rotation), p=0.5),
            Augmentation(gaussSample, p=0.5),
            Augmentation(cutout, p=0.5),
            Augmentation(OneOf(upsample, downsample), p=0.5),
        ]

train_ds = ImprovedKeypointsDataset(
    os.path.join(DATA_DIR, f'{DATASET}_train_keypoints'),
    use_augs=True, label_map=label_map, mode='train'
)
val_ds = KeypointsDataset(
    os.path.join(DATA_DIR, f'{DATASET}_val_keypoints'),
    use_augs=False, label_map=label_map, mode='val'
)
train_loader = torch_data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = torch_data.DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

# ── Optimizer: AdamW with weight decay ────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# ── Scheduler: Linear warmup + CosineAnnealing ────────────────────────
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Mixup helper ──────────────────────────────────────────────────────
def mixup_batch(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

# ── Training loop ─────────────────────────────────────────────────────
save_file = os.path.join(SAVE_PATH, f'include_no_cnn_transformer_{SIZE}_improved.pth')
early_stopping = EarlyStopping(patience=PATIENCE, mode='max')
best_val_acc = 0

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_losses = AverageMeter(); train_accs = AverageMeter()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    for batch in pbar:
        x, y = batch['data'].to(device), batch['label'].to(device)
        x, y_a, y_b, lam = mixup_batch(x, y, MIXUP_ALPHA)
        optimizer.zero_grad()
        preds = model(x)
        loss = lam * F.cross_entropy(preds, y_a, label_smoothing=0.1) + \
               (1 - lam) * F.cross_entropy(preds, y_b, label_smoothing=0.1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        train_losses.update(loss.item())
        acc = accuracy_score(y.cpu().numpy(), preds.detach().cpu().argmax(-1).numpy())
        train_accs.update(acc)
        pbar.set_postfix(loss=f'{train_losses.avg:.4f}', acc=f'{train_accs.avg:.4f}', lr=f'{scheduler.get_last_lr()[0]:.6f}')
    scheduler.step()

    # Validate
    model.eval()
    val_losses = AverageMeter(); val_accs = AverageMeter()
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]'):
            x, y = batch['data'].to(device), batch['label'].to(device)
            preds = model(x)
            loss = F.cross_entropy(preds, y)
            val_losses.update(loss.item())
            acc = accuracy_score(y.cpu().numpy(), preds.cpu().argmax(-1).numpy())
            val_accs.update(acc)

    print(f'Epoch {epoch+1}: train_loss={train_losses.avg:.4f} train_acc={train_accs.avg:.4f} | val_loss={val_losses.avg:.4f} val_acc={val_accs.avg:.4f}')

    if val_accs.avg > best_val_acc:
        best_val_acc = val_accs.avg
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'score': best_val_acc}, save_file)
        print(f'  Saved best model: {best_val_acc:.4f}')

    early_stopping(save_file, val_accs.avg, model, optimizer, scheduler)
    if early_stopping.early_stop:
        print('Early stopping triggered')
        break

print(f'\nBest val accuracy: {best_val_acc:.4f} ({round(best_val_acc*100,2)}%)')

In [ ]:
# ── Step 5: Test accuracy ─────────────────────────────────────────────
import glob as glob_module
import pandas as pd

cp = torch.load(save_file, map_location='cpu', weights_only=False)
print('Val accuracy:', round(cp['score'] * 100, 2), '%')

model.load_state_dict(cp['model'])
model.eval()

idx_to_label = {v: k for k, v in label_map.items()}
correct = 0; total = 0
test_ds = KeypointsDataset(
    os.path.join(DATA_DIR, f'{DATASET}_test_keypoints'),
    use_augs=False, label_map=label_map, mode='test'
)
test_loader = torch_data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing'):
        x, y = batch['data'].to(device), batch['label'].to(device)
        preds = model(x).cpu().argmax(-1)
        correct += (preds == y.cpu()).sum().item()
        total += y.size(0)

print(f'Test accuracy: {correct}/{total} = {round(100*correct/total, 2)}%')

In [ ]:
# ── Step 6: Download the trained model ───────────────────────────────
from google.colab import files
files.download(save_file)
print('Downloaded:', save_file)
print()
print('Next steps:')
print('1. Place in Major_Project/INCLUDE/')
print('2. Rename to: include_no_cnn_transformer_small.pth')
print('3. unified_app.py already uses this file — sign recognition will work!')